# Lab 5 — Value-based Reinforcement Learning (DQN)
CartPole DQN / Atari Pong DQN / Enhanced DQN

## 使用前準備
1. 執行階段 → 變更執行階段類型 → **GPU (T4)**
2. Colab Secrets 中設定 `WANDB_API_KEY`（左側鑰匙圖示）
3. 首次執行：依序跑 Section 1–5，之後每次重連只需跑 Section 1–5 即可
4. 模型自動備份至 `MyDrive/lab5/`，斷線後可從 checkpoint 繼續

## 1. 確認 GPU

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
Memory: 23.7 GB


## 2. 安裝套件

In [2]:
!pip install -q gymnasium==1.1.1 ale-py opencv-python-headless wandb imageio imageio-ffmpeg
!pip install -q 'gymnasium[atari]' autorom
!python -m autorom --accept-license
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.4/965.4 kB 58.5 MB/s eta 0:00:00
/usr/bin/python3: No module named autorom
Done.


## 3. Clone / Pull Repo

In [3]:
import os

REPO_PATH = '/content/NYCU-Deep-Learing'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/InnisChen/NYCU-Deep-Learing {REPO_PATH}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_PATH} pull

%cd {REPO_PATH}/lab5
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/NYCU-Deep-Learing'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 385 (delta 50), reused 105 (delta 42), pack-reused 264 (from 1)
Receiving objects: 100% (385/385), 2.43 MiB | 33.18 MiB/s, done.
Resolving deltas: 100% (192/192), done.
/content/NYCU-Deep-Learing/lab5
Working directory: /content/NYCU-Deep-Learing/lab5


## 4. 掛載 Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_LAB5 = '/content/drive/MyDrive/lab5'
DRIVE_CKPT = f'{DRIVE_LAB5}/checkpoints'
os.makedirs(DRIVE_LAB5, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)
print(f'Drive ready: {DRIVE_LAB5}')

Mounted at /content/drive
Drive ready: /content/drive/MyDrive/lab5


## 5. Wandb 登入

In [5]:
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))
print('Wandb login OK')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: innis (innis-national-taiwan-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login OK


---
## 6. Task 1 — CartPole DQN
目標：平均 reward > 480（滿分門檻）

In [ ]:
# ── Task 1 CONFIG（可自由修改）────────────────────────────────────────
T1_EPISODES       = 2000
T1_BATCH_SIZE     = 32
T1_LR             = 1e-4
T1_MEMORY_SIZE    = 50000
T1_REPLAY_START   = 1000    # 開始訓練前需收集的 transitions 數
T1_EPSILON_DECAY  = 0.9995
T1_EPSILON_MIN    = 0.05
T1_TARGET_UPDATE  = 500
T1_CHECKPOINT_FREQ= 0       # 0 = 不存 checkpoint（CartPole 跑很快）
T1_SAVE_DIR       = '/content/results_task1'
T1_WANDB_NAME     = 'task1-cartpole'
T1_RESUME         = False   # True = 從上次中斷繼續
STUDENT_ID        = 'B11107027'

In [ ]:
# ── Task 1 訓練 ───────────────────────────────────────────────────────
import shutil, os

os.makedirs(T1_SAVE_DIR, exist_ok=True)

resume_flag = ''
if T1_RESUME:
    ckpt_src = f'{DRIVE_CKPT}/task1_checkpoint.pt'
    ckpt_dst = f'{T1_SAVE_DIR}/checkpoint.pt'
    if os.path.exists(ckpt_src):
        shutil.copy(ckpt_src, ckpt_dst)
        resume_flag = f'--resume {ckpt_dst}'
        print(f'Loaded checkpoint from Drive')
    else:
        print('No checkpoint found, starting fresh')

cmd = (f'python dqn.py'
       f' --task 1'
       f' --student-id {STUDENT_ID}'
       f' --env-name CartPole-v1'
       f' --episodes {T1_EPISODES}'
       f' --save-dir {T1_SAVE_DIR}'
       f' --wandb-run-name {T1_WANDB_NAME}'
       f' --batch-size {T1_BATCH_SIZE}'
       f' --memory-size {T1_MEMORY_SIZE}'
       f' --lr {T1_LR}'
       f' --epsilon-decay {T1_EPSILON_DECAY}'
       f' --epsilon-min {T1_EPSILON_MIN}'
       f' --replay-start-size {T1_REPLAY_START}'
       f' --target-update-frequency {T1_TARGET_UPDATE}'
       f' --checkpoint-freq {T1_CHECKPOINT_FREQ}'
       f' {resume_flag}')
print('Command:', cmd)
!{cmd}

# 訓練完成後備份至 Drive
best_model = f'{T1_SAVE_DIR}/LAB5_{STUDENT_ID}_task1.pt'
if os.path.exists(best_model):
    shutil.copy(best_model, f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task1.pt')
    print(f'Backed up Task 1 model to Drive')

In [ ]:
# ── Task 1 評估（CartPole，20 seeds）────────────────────────────────
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import sys
sys.path.insert(0, '.')
from dqn import DQN, AtariPreprocessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task1.pt'

env = gym.make('CartPole-v1')
preprocessor = AtariPreprocessor()
state_dim = env.observation_space.shape[0]
num_actions = env.action_space.n

model = DQN(num_actions, state_dim=state_dim, use_cnn=False).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

rewards = []
for seed in range(20):
    obs, _ = env.reset(seed=seed)
    state = preprocessor.reset(obs)
    done = False
    total_reward = 0
    while not done:
        state_tensor = torch.from_numpy(np.array(state)).float().unsqueeze(0).to(device)
        with torch.no_grad():
            action = model(state_tensor).argmax().item()
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        state = preprocessor.step(obs)
    rewards.append(total_reward)
    print(f'Seed {seed:2d}: reward = {total_reward:.1f}')

print(f'\nAverage (20 seeds): {np.mean(rewards):.2f}  |  Min: {np.min(rewards):.1f}  Max: {np.max(rewards):.1f}')
env.close()

---
## 7. Task 2 — Atari Pong DQN
目標：平均 reward > 19（滿分門檻）  
預估訓練時間：~1M steps，T4 約 8–15 小時

In [ ]:
# ── Task 2 CONFIG（可自由修改）────────────────────────────────────────
STUDENT_ID         = 'B11107027'
T2_EPISODES        = 10000
T2_BATCH_SIZE      = 64
T2_LR              = 1e-4
T2_MEMORY_SIZE     = 200000
T2_REPLAY_START    = 50000
T2_EPSILON_DECAY   = 0.999995
T2_EPSILON_MIN     = 0.05
T2_TARGET_UPDATE   = 5000
T2_CHECKPOINT_FREQ = 50000
T2_SAVE_DIR        = '/content/results_task2'
T2_WANDB_NAME      = 'task2-pong'
T2_RESUME          = False    # 斷線後改成 True 繼續訓練

In [ ]:
# ── Task 2 訓練 ───────────────────────────────────────────────────────
import shutil, os

os.makedirs(T2_SAVE_DIR, exist_ok=True)

# 讓 dqn.py 的 save_checkpoint() 自動備份到 Drive
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_flag = ''
if T2_RESUME:
    ckpt_src = f'{DRIVE_CKPT}/task2_checkpoint.pt'
    ckpt_dst = f'{T2_SAVE_DIR}/checkpoint.pt'
    if os.path.exists(ckpt_src):
        shutil.copy(ckpt_src, ckpt_dst)
        resume_flag = f'--resume {ckpt_dst}'
        print(f'Loaded checkpoint from Drive')
    else:
        print('No checkpoint found, starting fresh')

cmd = (f'python dqn.py'
       f' --task 2'
       f' --student-id {STUDENT_ID}'
       f' --env-name ALE/Pong-v5'
       f' --episodes {T2_EPISODES}'
       f' --save-dir {T2_SAVE_DIR}'
       f' --wandb-run-name {T2_WANDB_NAME}'
       f' --batch-size {T2_BATCH_SIZE}'
       f' --memory-size {T2_MEMORY_SIZE}'
       f' --lr {T2_LR}'
       f' --epsilon-decay {T2_EPSILON_DECAY}'
       f' --epsilon-min {T2_EPSILON_MIN}'
       f' --replay-start-size {T2_REPLAY_START}'
       f' --target-update-frequency {T2_TARGET_UPDATE}'
       f' --checkpoint-freq {T2_CHECKPOINT_FREQ}'
       f' {resume_flag}')
print('Command:', cmd)
!{cmd}

In [ ]:
# ── Task 2 評估（Pong，seeds 0~19）───────────────────────────────────
import shutil, os

EVAL_MODEL = f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task2.pt'
EVAL_DIR   = '/content/eval_task2_videos'

# 把 Drive 上的模型複製到本地
shutil.copy(EVAL_MODEL, f'LAB5_{STUDENT_ID}_task2.pt')

!python test_model.py \
    --model-path LAB5_{STUDENT_ID}_task2.pt \
    --output-dir {EVAL_DIR} \
    --episodes 20 \
    --seed 0

---
## 8. Task 3 — Enhanced DQN（Double DQN + PER）
目標：越早達到 score 19 分數越高  
里程碑快照自動儲存：600k / 1M / 1.5M / 2M / 2.5M steps

In [8]:
# ── Task 3 CONFIG（可自由修改）────────────────────────────────────────
STUDENT_ID         = 'B11107027'
T3_EPISODES        = 10000
T3_BATCH_SIZE      = 32       # 32 → 64
T3_LR              = 0.0003
T3_MEMORY_SIZE     = 200000
T3_REPLAY_START    = 20000
T3_EPSILON_DECAY_TYPE    = 'linear'
T3_EPSILON_DECAY_STEPS   = 500000      # 500k steps 完成 decay
T3_EPSILON_MIN           = 0.01
#T3_EPSILON_DECAY   = 0.999995
T3_TARGET_UPDATE   = 2000
T3_CHECKPOINT_FREQ = 50000
T3_SAVE_DIR        = '/content/results_task3'
T3_WANDB_NAME      = 'task3-pong-enhanced'
T3_RESUME          = False

# Task 3 enhancements
T3_USE_PER               = True
T3_USE_DOUBLE            = True
T3_N_STEP                = 3
T3_PER_ALPHA             = 0.6
T3_PER_BETA              = 0.4
T3_PER_BETA_ANNEAL_STEPS = 2000000

In [ ]:
# ── Task 3 訓練 ───────────────────────────────────────────────────────
import shutil, os

os.makedirs(T3_SAVE_DIR, exist_ok=True)

resume_flag = ''
if T3_RESUME:
    ckpt_src = f'{DRIVE_CKPT}/task3_checkpoint.pt'
    ckpt_dst = f'{T3_SAVE_DIR}/checkpoint.pt'
    if os.path.exists(ckpt_src):
        shutil.copy(ckpt_src, ckpt_dst)
        resume_flag = f'--resume {ckpt_dst}'
        print('Loaded checkpoint from Drive')
    else:
        print('No checkpoint found, starting fresh')

per_flag    = '--use-per'    if T3_USE_PER    else ''
double_flag = '--use-double' if T3_USE_DOUBLE else ''

cmd = (f'DRIVE_CKPT_DIR={DRIVE_CKPT} python dqn.py'
       f' --task 3'
       f' --student-id {STUDENT_ID}'
       f' --env-name ALE/Pong-v5'
       f' --episodes {T3_EPISODES}'
       f' --save-dir {T3_SAVE_DIR}'
       f' --wandb-run-name {T3_WANDB_NAME}'
       f' --batch-size {T3_BATCH_SIZE}'
       f' --memory-size {T3_MEMORY_SIZE}'
       f' --lr {T3_LR}'
       f' --epsilon-decay-type {T3_EPSILON_DECAY_TYPE}'
       f' --epsilon-decay-steps {T3_EPSILON_DECAY_STEPS}'
       f' --epsilon-min {T3_EPSILON_MIN}'
       f' --replay-start-size {T3_REPLAY_START}'
       f' --target-update-frequency {T3_TARGET_UPDATE}'
       f' --checkpoint-freq {T3_CHECKPOINT_FREQ}'
       f' {per_flag} {double_flag}'
       f' --n-step {T3_N_STEP}'
       f' --per-alpha {T3_PER_ALPHA}'
       f' --per-beta {T3_PER_BETA}'
       f' --per-beta-anneal-steps {T3_PER_BETA_ANNEAL_STEPS}'
       f' {resume_flag}')
print('Command:', cmd)
!{cmd}

Command: DRIVE_CKPT_DIR=/content/drive/MyDrive/lab5/checkpoints python dqn.py --task 3 --student-id B11107027 --env-name ALE/Pong-v5 --episodes 10000 --save-dir /content/results_task3 --wandb-run-name task3-pong-enhanced --batch-size 32 --memory-size 200000 --lr 0.0003 --epsilon-decay-type linear --epsilon-decay-steps 500000 --epsilon-min 0.01 --replay-start-size 20000 --target-update-frequency 2000 --checkpoint-freq 50000 --use-per --use-double --n-step 3 --per-alpha 0.6 --per-beta 0.4 --per-beta-anneal-steps 2000000 
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: innis (innis-national-taiwan-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run xar4k95r (0.2s)
wandb: ⣽ setting up run xar4k95r (0.2s)
wandb: ⣾ setting up run xar4k95r (0.2s)
wandb: ⣷ setting up run xar4k95r (0.2s)
wandb: ⣯ s

In [9]:
# ── Task 3 訓練中斷時：備份 checkpoint + 已完成里程碑到 Drive（手動執行）
import shutil, os

T3_SAVE_DIR = '/content/results_task3'

ckpt_src = f'{T3_SAVE_DIR}/checkpoint.pt'
if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, f'{DRIVE_CKPT}/task3_checkpoint.pt')
    print('Checkpoint backed up')

for milestone in [600000, 1000000, 1500000, 2000000, 2500000]:
    src = f'{T3_SAVE_DIR}/LAB5_{STUDENT_ID}_task3_{milestone}.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task3_{milestone}.pt')
        print(f'Milestone {milestone} backed up')

In [ ]:
# ── Task 3 評估所有里程碑 ─────────────────────────────────────────────
import shutil, os

MILESTONES = [600000, 1000000, 1500000, 2000000, 2500000]

for milestone in MILESTONES:
    model_drive = f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task3_{milestone}.pt'
    if not os.path.exists(model_drive):
        print(f'SKIP {milestone}: not found in Drive')
        continue
    local_model = f'LAB5_{STUDENT_ID}_task3_{milestone}.pt'
    shutil.copy(model_drive, local_model)

    eval_dir = f'/content/eval_task3_{milestone}'
    print(f'\n=== Evaluating {milestone} steps ===')
    !python test_model.py \
        --model-path {local_model} \
        --output-dir {eval_dir} \
        --episodes 20 \
        --seed 0

In [ ]:
# ── Task 3 best model 評估 ────────────────────────────────────────────
import shutil

best_drive = f'{DRIVE_LAB5}/LAB5_{STUDENT_ID}_task3_best.pt'
local_best = f'LAB5_{STUDENT_ID}_task3_best.pt'
shutil.copy(best_drive, local_best)

!python test_model.py \
    --model-path {local_best} \
    --output-dir /content/eval_task3_best \
    --episodes 20 \
    --seed 0

---
## 10. 釋放 GPU Runtime

In [ ]:
# 確認所有模型都已備份到 Drive 再執行
from google.colab import runtime
runtime.unassign()